In [ ]:
import os
# protobufの互換性問題を回避するための環境変数設定
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from urllib.error import URLError

import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, KFold, GroupKFold
from sklearn.metrics import make_scorer
from sklearn.multioutput import MultiOutputRegressor

import torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
from PIL import Image
from transformers import AutoModel
import timm

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [2]:
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
WEIGHTS = np.array([0.1, 0.1, 0.1, 0.2, 0.5])

def weighted_r2_metric(y_true, y_pred):
    ss_res = np.sum(WEIGHTS * np.sum((y_true - y_pred)**2, axis=0))
    global_mean = np.average(np.mean(y_true, axis=0), weights=WEIGHTS)
    ss_tot = np.sum(WEIGHTS * np.sum((y_true - global_mean)**2, axis=0))
    return 1 - (ss_res / ss_tot)

weighted_scorer = make_scorer(weighted_r2_metric, greater_is_better=True)

In [3]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

@torch.no_grad()
def extract_dense_features(df, model, device, base_path):
    """TTA 提取 dense 特征 -> DataFrame"""
    tta_transforms = [
        T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]),
        T.Compose([T.Resize(256), T.CenterCrop(224), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(MEAN, STD)])
    ]
    unique_paths = df['image_path'].unique()
    all_feats = []
    for path in tqdm(unique_paths, desc="GPU Feature Extraction"):
        img = Image.open(base_path / path).convert("RGB")
        tta_results = [
            model(aug(img).unsqueeze(0).to(device)).last_hidden_state[:, 1:, :].mean(dim=1).cpu().numpy()
            for aug in tta_transforms
        ]
        all_feats.append(np.mean(tta_results, axis=0))
    feat_matrix = np.vstack(all_feats)
    feat_cols = [f"feat_{i}" for i in range(feat_matrix.shape[1])]
    return pd.DataFrame(feat_matrix, columns=feat_cols), unique_paths

In [ ]:
@torch.no_grad()
def extract_efficientnet_features(df, model, device, base_path):
    """EfficientNetでTTA特徴量抽出 -> DataFrame"""
    tta_transforms = [
        T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]),
        T.Compose([T.Resize(256), T.CenterCrop(224), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(MEAN, STD)])
    ]
    unique_paths = df['image_path'].unique()
    all_feats = []
    for path in tqdm(unique_paths, desc="EfficientNet Feature Extraction"):
        img = Image.open(base_path / path).convert("RGB")
        tta_results = []
        for aug in tta_transforms:
            img_tensor = aug(img).unsqueeze(0).to(device)
            # EfficientNetの特徴量を抽出（最終層の前）
            features = model.forward_features(img_tensor)
            # Global Average Pooling
            if len(features.shape) > 2:
                features = features.mean(dim=[2, 3])  # Spatial dimensions
            tta_results.append(features.cpu().numpy())
        all_feats.append(np.mean(tta_results, axis=0))
    feat_matrix = np.vstack(all_feats)
    feat_cols = [f"effnet_feat_{i}" for i in range(feat_matrix.shape[1])]
    return pd.DataFrame(feat_matrix, columns=feat_cols), unique_paths



In [ ]:
def create_image_groups(df, date_tolerance=0):
    """
    日付と場所で画像をグループ化
    
    Parameters:
    -----------
    df : pd.DataFrame
        元のtrain.csv
    date_tolerance : int
        日付の許容範囲（日数）。0=同一日のみ、1=±1日など
    
    Returns:
    --------
    group_ids : dict
        image_path -> group_id のマッピング
    group_info : pd.DataFrame
        グループ情報を含むDataFrame
    """
    # Sampling_Dateをdatetime型に変換
    df = df.copy()
    df['Sampling_Date'] = pd.to_datetime(df['Sampling_Date'])
    
    # image_pathごとに最初の行の日付とStateを取得
    image_meta = df.groupby('image_path').agg({
        'Sampling_Date': 'first',
        'State': 'first'
    }).reset_index()
    
    # 日付を正規化（tolerance考慮）
    if date_tolerance == 0:
        image_meta['date_key'] = image_meta['Sampling_Date'].dt.date.astype(str)
    else:
        # 日付をtolerance範囲で丸める
        image_meta['date_normalized'] = (
            image_meta['Sampling_Date'].dt.floor(f'{date_tolerance+1}D')
        ).dt.date.astype(str)
        image_meta['date_key'] = image_meta['date_normalized']
    
    # グループIDを作成（State + date_key）
    image_meta['group_id'] = image_meta['State'].astype(str) + '_' + image_meta['date_key']
    
    # グループIDを数値に変換
    unique_groups = image_meta['group_id'].unique()
    group_id_map = {group: idx for idx, group in enumerate(unique_groups)}
    image_meta['group_id_num'] = image_meta['group_id'].map(group_id_map)
    
    # image_path -> group_id のマッピングを作成
    group_ids = dict(zip(image_meta['image_path'], image_meta['group_id_num']))
    
    return group_ids, image_meta



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_path = Path("/kaggle/input/csiro-biomass")
model_dir = os.path.abspath("/kaggle/input/dinov2/pytorch/large/1")

# DINOv2モデル（既存）
model = AutoModel.from_pretrained(
    model_dir, local_files_only=True, trust_remote_code=True
).to(device).eval()

# EfficientNet-B4モデルを追加（エラーハンドリング付き）
try:
    effnet_model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
    print("EfficientNet-B4: Pretrained weights loaded successfully")
except (URLError, OSError, Exception) as e:
    if "name resolution" in str(e).lower() or "temporary failure" in str(e).lower() or "gaierror" in str(type(e).__name__).lower():
        print(f"Warning: Failed to download EfficientNet-B4 weights: {e}")
        print("Initializing EfficientNet-B4 without pretrained weights (performance may be reduced)")
        effnet_model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=0)
    else:
        raise

effnet_model = effnet_model.to(device).eval()

train_df = pd.read_csv(base_path / "train.csv")

# 画像を日付・場所でグループ化
DATE_TOLERANCE = 0  # 0=同一日のみ、変更可能
group_ids, group_info = create_image_groups(train_df, date_tolerance=DATE_TOLERANCE)

print(f"Total training images: {train_df['image_path'].nunique()}")
print(f"Total groups: {group_info['group_id_num'].nunique()}")
print(f"Average images per group: {group_info.groupby('group_id_num').size().mean():.2f}")

# すべてのデータを使用
train_p = train_df.pivot_table(
    index="image_path", columns="target_name", values="target"
).reset_index()

# グループID配列を作成（X_train_dfと同じ順序）
train_groups = np.array([group_ids[path] for path in train_p['image_path']])

# DINOv2特徴量
X_train_dinov2, _ = extract_dense_features(train_p, model, device, base_path)

# EfficientNet特徴量
X_train_effnet, _ = extract_efficientnet_features(train_p, effnet_model, device, base_path)

# 特徴量を結合
X_train_df = pd.concat([X_train_dinov2, X_train_effnet], axis=1)

print(f"DINOv2特徴量数: {X_train_dinov2.shape[1]}")
print(f"EfficientNet特徴量数: {X_train_effnet.shape[1]}")
print(f"結合後特徴量数: {X_train_df.shape[1]}")

Y_train = train_p[TARGETS].values

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_df), columns=X_train_df.columns
)

2026-01-04 16:25:24.069626: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767543924.271222      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767543924.329984      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767543924.812061      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767543924.812104      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767543924.812107      24 computation_placer.cc:177] computation placer alr

In [ ]:
param_grid = {
    "estimator__objective": ['gamma', 'tweedie'],  # 右に裾を引く分布に対応
    "estimator__learning_rate": [0.05, 0.1],
    "estimator__num_leaves": [31, 63],
    "estimator__n_estimators": [100, 200],
    "estimator__random_state": [42]
}
base_lgbm = MultiOutputRegressor(LGBMRegressor(verbosity=-1))
grid_search = GridSearchCV(
    base_lgbm,
    param_grid,
    cv=GroupKFold(n_splits=5),  # shuffleは不要（グループが自動で分離される）
    scoring=weighted_scorer,
    n_jobs=-1,
    verbose=0
)

try:
    grid_search.fit(X_train_scaled, Y_train, groups=train_groups)
    print("\n" + "="*50)
    print(f"Best LGBM Params: {grid_search.best_params_}")
    print(f"Best CV Weighted R2: {grid_search.best_score_:.4f}")
    print("="*50)
except AttributeError as e:
    if "GetPrototype" in str(e):
        print(f"Warning: protobuf compatibility issue detected: {e}")
        print("Attempting to continue with default parameters...")
        # フォールバック: デフォルトパラメータでモデルを訓練
        fallback_lgbm = MultiOutputRegressor(LGBMRegressor(
            objective='gamma',  # 右に裾を引く分布に対応
            learning_rate=0.05,
            n_estimators=200,
            num_leaves=31,
            random_state=42,
            verbosity=-1
        ))
        fallback_lgbm.fit(X_train_scaled, Y_train)
        # grid_searchオブジェクトをフォールバックモデルで置き換え
        grid_search.best_estimator_ = fallback_lgbm
        grid_search.best_params_ = {"estimator__objective": "gamma", "estimator__learning_rate": 0.05, "estimator__n_estimators": 200, "estimator__num_leaves": 31, "estimator__random_state": 42}
        print("\n" + "="*50)
        print("Using default parameters due to compatibility issue")
        print("="*50)
    else:
        raise
except Exception as e:
    print(f"Error during GridSearchCV: {e}")
    raise


Best LGBM Params: {'estimator__learning_rate': 0.05, 'estimator__n_estimators': 200, 'estimator__num_leaves': 31, 'estimator__random_state': 42}
Best CV Weighted R2: 0.8145


In [ ]:
# 各foldの詳細スコア分析
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

# 各foldのスコアを計算
cv_scores = cross_val_score(
    grid_search.best_estimator_,
    X_train_scaled,
    Y_train,
    groups=train_groups,
    cv=GroupKFold(n_splits=5),
    scoring=weighted_scorer,
    n_jobs=-1
)

print("="*50)
print("CV Fold詳細スコア:")
print(f"Fold 1: {cv_scores[0]:.4f}")
print(f"Fold 2: {cv_scores[1]:.4f}")
print(f"Fold 3: {cv_scores[2]:.4f}")
print(f"Fold 4: {cv_scores[3]:.4f}")
print(f"Fold 5: {cv_scores[4]:.4f}")
print(f"Mean: {cv_scores.mean():.4f}")
print(f"Std: {cv_scores.std():.4f}")
print(f"Min: {cv_scores.min():.4f}")
print(f"Max: {cv_scores.max():.4f}")
print("="*50)



In [ ]:
# グループ分布の分析
group_sizes = pd.Series(train_groups).value_counts().sort_index()
print("="*50)
print("グループサイズ統計:")
print(f"最小グループサイズ: {group_sizes.min()}")
print(f"最大グループサイズ: {group_sizes.max()}")
print(f"平均グループサイズ: {group_sizes.mean():.2f}")
print(f"中央値グループサイズ: {group_sizes.median():.2f}")
print(f"グループサイズの標準偏差: {group_sizes.std():.2f}")
print("="*50)

# グループごとのターゲット値の統計
group_target_stats = []
for group_id in np.unique(train_groups):
    group_mask = train_groups == group_id
    group_targets = Y_train[group_mask]
    group_target_stats.append({
        'group_id': group_id,
        'size': group_mask.sum(),
        'mean_total': group_targets[:, -1].mean(),  # Dry_Total_g
        'std_total': group_targets[:, -1].std()
    })

group_stats_df = pd.DataFrame(group_target_stats)
print("\nグループごとのターゲット値統計:")
print(group_stats_df.describe())



In [ ]:
# 予測値の分布分析
train_preds = grid_search.best_estimator_.predict(X_train_scaled)
train_preds = np.maximum(train_preds, 0)

print("="*50)
print("訓練データでの予測値と実際の値の比較:")
for i, target_name in enumerate(TARGETS):
    print(f"\n{target_name}:")
    print(f"  実際の値 - Mean: {Y_train[:, i].mean():.2f}, Std: {Y_train[:, i].std():.2f}")
    print(f"  予測値   - Mean: {train_preds[:, i].mean():.2f}, Std: {train_preds[:, i].std():.2f}")
    print(f"  相関係数: {np.corrcoef(Y_train[:, i], train_preds[:, i])[0, 1]:.4f}")

# テストデータの予測値は後で計算されるので、ここではコメントアウト
# print("\nテストデータでの予測値統計:")
# for i, target_name in enumerate(TARGETS):
#     print(f"{target_name}: Mean={avg_preds[:, i].mean():.2f}, Std={avg_preds[:, i].std():.2f}")
print("="*50)



In [ ]:
# 特徴量重要度の分析
feature_importance = []
for i, estimator in enumerate(grid_search.best_estimator_.estimators_):
    importance = estimator.feature_importances_
    feature_importance.append(importance)

# 平均重要度を計算
mean_importance = np.mean(feature_importance, axis=0)
feature_importance_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': mean_importance
}).sort_values('importance', ascending=False)

print("="*50)
print("Top 20 特徴量重要度:")
print(feature_importance_df.head(20))
print("="*50)



In [ ]:
# 過学習の兆候チェック
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    grid_search.best_estimator_,
    X_train_scaled,
    Y_train,
    groups=train_groups,
    cv=GroupKFold(n_splits=5),
    scoring=weighted_scorer,
    return_train_score=True,
    n_jobs=-1
)

print("="*50)
print("過学習チェック:")
print(f"訓練スコア平均: {cv_results['train_score'].mean():.4f} ± {cv_results['train_score'].std():.4f}")
print(f"検証スコア平均: {cv_results['test_score'].mean():.4f} ± {cv_results['test_score'].std():.4f}")
print(f"スコア差: {cv_results['train_score'].mean() - cv_results['test_score'].mean():.4f}")
print("="*50)



In [ ]:
test_df = pd.read_csv(base_path / "test.csv")
test_uniq = pd.DataFrame({"image_path": test_df["image_path"].unique()})

# DINOv2特徴量
X_test_dinov2, test_ids = extract_dense_features(test_uniq, model, device, base_path)

# EfficientNet特徴量
X_test_effnet, _ = extract_efficientnet_features(test_uniq, effnet_model, device, base_path)

# 特徴量を結合
X_test_df = pd.concat([X_test_dinov2, X_test_effnet], axis=1)

X_test_scaled = pd.DataFrame(scaler.transform(X_test_df), columns=X_test_df.columns)

test_preds = grid_search.best_estimator_.predict(X_test_scaled)
avg_preds = np.maximum(test_preds, 0)

GPU Feature Extraction: 100%|██████████| 1/1 [00:00<00:00,  4.25it/s]


In [7]:
final_rows = []
for i, path in enumerate(test_ids):
    image_id = Path(path).stem
    for j, target_name in enumerate(TARGETS):
        final_rows.append(
            {"sample_id": f"{image_id}__{target_name}", "target": avg_preds[i, j]}
        )
pd.DataFrame(final_rows).to_csv("submission.csv", index=False)
print("\n🚀 LGBM submission saved to submission.csv")


🚀 LGBM submission saved to submission.csv
